In [1]:
from google.colab import drive
drive.mount('/content/drive')

import torch
import json
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import faiss
import time
import os

DATA_PATH = "/content/drive/MyDrive/medrag/pubmedqa_filtered.json"
OUTPUT_DIR = "/content/drive/MyDrive/medrag/biomistral_paired_passes"
os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(DATA_PATH, "r") as f:
    corpus = json.load(f)

print(f"Corpus loaded: {len(corpus)} samples")
print(f"Output directory: {OUTPUT_DIR}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

Mounted at /content/drive
Corpus loaded: 759 samples
Output directory: /content/drive/MyDrive/medrag/biomistral_paired_passes
GPU: NVIDIA A100-SXM4-40GB


In [3]:
MODEL_ID = "BioMistral/BioMistral-7B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    attn_implementation="eager",
    use_safetensors=False
)

model = model.to("cuda")
model.eval()

for param in model.parameters():
    param.requires_grad = False

n_layers = model.config.num_hidden_layers
hidden_size = model.config.hidden_size
vocab_size = model.config.vocab_size

print(f"Model loaded: {n_layers} layers, hidden size {hidden_size}")

Model loaded: 32 layers, hidden size 4096


In [5]:
documents = []
doc_metadata = []

for sample in corpus:
    for abstract in sample["supporting_abstracts"]:
        documents.append(abstract)
        doc_metadata.append({
            "pubid": sample["pubid"],
            "query": sample["query"],
            "gold_answer": sample["gold_answer"],
            "label": sample["label"]
        })

tokenized_docs = [doc.lower().split() for doc in documents]
bm25 = BM25Okapi(tokenized_docs)

enc_model = SentenceTransformer("NeuML/pubmedbert-base-embeddings")
doc_embeddings = enc_model.encode(
    documents,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

faiss.normalize_L2(doc_embeddings)
index = faiss.IndexFlatIP(doc_embeddings.shape[1])
index.add(doc_embeddings)

print(f"Retriever ready. {index.ntotal} documents indexed.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/80 [00:00<?, ?it/s]

Retriever ready. 2542 documents indexed.


In [6]:
def bm25_retrieve(query, k=3):
    scores = bm25.get_scores(query.lower().split())
    top_k = np.argsort(scores)[::-1][:k]
    return [{"abstract": documents[i], "pubid": doc_metadata[i]["pubid"],
             "score": float(scores[i]), "gold_answer": doc_metadata[i]["gold_answer"],
             "label": doc_metadata[i]["label"]} for i in top_k]

def faiss_retrieve(query, k=3):
    qe = enc_model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(qe)
    scores, indices = index.search(qe, k)
    return [{"abstract": documents[i], "pubid": doc_metadata[i]["pubid"],
             "score": float(s), "gold_answer": doc_metadata[i]["gold_answer"],
             "label": doc_metadata[i]["label"]} for s, i in zip(scores[0], indices[0])]

def hybrid_retrieve(query, k=3, rrf_k=60):
    bm25_results = bm25_retrieve(query, k=k*2)
    faiss_results = faiss_retrieve(query, k=k*2)
    combined = {}

    for rank, r in enumerate(bm25_results):
        combined[r["abstract"]] = {"meta": r, "score": 1 / (rrf_k + rank + 1)}

    for rank, r in enumerate(faiss_results):
        key = r["abstract"]
        if key in combined:
            combined[key]["score"] += 1 / (rrf_k + rank + 1)
        else:
            combined[key] = {"meta": r, "score": 1 / (rrf_k + rank + 1)}

    return [v["meta"] for v in sorted(combined.values(),
            key=lambda x: x["score"], reverse=True)[:k]]

def build_rag_prompt(query, k=3):
    results = hybrid_retrieve(query, k=k)
    context_parts = [f"[{i+1}] {r['abstract']}" for i, r in enumerate(results)]
    context = "\n\n".join(context_parts)
    prompt = f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"
    return prompt, results

def build_suppressed_prompt(query):
    return f"Question: {query}\nAnswer:"

print("Retrieval and prompt functions ready.")

Retrieval and prompt functions ready.


In [7]:
hook_storage = {
    "hidden_states": {},
    "attention_maps": {},
    "logits_per_layer": {}
}
hooks = []

def make_hidden_state_hook(layer_idx):
    def hook(module, input, output):
        hidden = output[0].detach().squeeze(0)
        hook_storage["hidden_states"][layer_idx] = hidden
        with torch.no_grad():
            normed = model.model.norm(hidden)
            logits = model.lm_head(normed)
        hook_storage["logits_per_layer"][layer_idx] = logits.detach()
    return hook

for i, block in enumerate(model.model.layers):
    h = block.register_forward_hook(make_hidden_state_hook(i))
    hooks.append(h)

print(f"Hooks registered on {n_layers} layers.")

Hooks registered on 32 layers.


In [8]:
def run_instrumented_pass(prompt, max_length=512):
    for key in hook_storage:
        hook_storage[key].clear()

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    ).to("cuda")

    seq_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model(**inputs, output_attentions=True)

    for i, attn in enumerate(outputs.attentions):
        hook_storage["attention_maps"][i] = attn.detach().squeeze(0)

    return {
        "hidden_states": {k: v.cpu().clone() for k, v in hook_storage["hidden_states"].items()},
        "attention_maps": {k: v.cpu().clone() for k, v in hook_storage["attention_maps"].items()},
        "logits_per_layer": {k: v.cpu().clone() for k, v in hook_storage["logits_per_layer"].items()},
        "seq_len": seq_len,
        "input_ids": inputs["input_ids"].detach().cpu().squeeze(0)
    }

print("Forward pass function ready.")

Forward pass function ready.


In [10]:
VALIDATION_SAMPLES = 20
divergence_log = []

print(f"Running paired forward passes on {VALIDATION_SAMPLES} samples...\n")

for idx, sample in enumerate(corpus[:VALIDATION_SAMPLES]):
    query = sample["query"]

    rag_prompt, retrieved = build_rag_prompt(query, k=3)
    suppressed_prompt = build_suppressed_prompt(query)

    rag_tensors = run_instrumented_pass(rag_prompt)
    sup_tensors = run_instrumented_pass(suppressed_prompt)

    rag_final_logits = rag_tensors["logits_per_layer"][n_layers-1][-1]
    sup_final_logits = sup_tensors["logits_per_layer"][n_layers-1][-1]

    rag_probs = torch.softmax(rag_final_logits.float(), dim=-1)
    sup_probs = torch.softmax(sup_final_logits.float(), dim=-1)

    kl = torch.sum(rag_probs * torch.log((rag_probs + 1e-10) / (sup_probs + 1e-10)))

    divergence_log.append({
        "idx": idx,
        "pubid": sample["pubid"],
        "query": query[:60],
        "rag_seq_len": rag_tensors["seq_len"],
        "sup_seq_len": sup_tensors["seq_len"],
        "kl_divergence": float(kl),
        "label": sample["label"],
        "retrieved_pubids": [r["pubid"] for r in retrieved]
    })

    if (idx + 1) % 5 == 0:
        print(f"  Completed {idx+1}/{VALIDATION_SAMPLES} samples...")

print("\n=== PAIRED PASS DIVERGENCE VALIDATION ===\n")
kl_values = [d["kl_divergence"] for d in divergence_log]
print(f"Mean KL divergence (RAG vs suppressed): {np.mean(kl_values):.4f}")
print(f"Min: {min(kl_values):.4f} | Max: {max(kl_values):.4f}")
print(f"\nAll samples show meaningful divergence: "
      f"{'YES' if np.mean(kl_values) > 0.01 else 'NO - check pipeline'}")

Running paired forward passes on 20 samples...

  Completed 5/20 samples...
  Completed 10/20 samples...
  Completed 15/20 samples...
  Completed 20/20 samples...

=== PAIRED PASS DIVERGENCE VALIDATION ===

Mean KL divergence (RAG vs suppressed): 2.1026
Min: 0.3443 | Max: 11.6801

All samples show meaningful divergence: YES


In [11]:
report_path = os.path.join(OUTPUT_DIR, "paired_pass_validation_report.json")

with open(report_path, "w") as f:
    json.dump({
        "model_id": MODEL_ID,
        "n_samples": VALIDATION_SAMPLES,
        "n_layers": n_layers,
        "hidden_size": hidden_size,
        "vocab_size": vocab_size,
        "mean_kl": float(np.mean(kl_values)),
        "min_kl": float(min(kl_values)),
        "max_kl": float(max(kl_values)),
        "all_diverge": bool(np.mean(kl_values) > 0.01),
        "samples": divergence_log
    }, f, indent=2)

for h in hooks:
    h.remove()

print(f"Validation report saved to {report_path}")
print("02_biomistral_paired_forward_pass complete")

Validation report saved to /content/drive/MyDrive/medrag/biomistral_paired_passes/paired_pass_validation_report.json
02_biomistral_paired_forward_pass complete
